# LangGraph を使用したエージェントの構築

## 学習目標

1. グラフの状態（State）の理解: LangGraph アプリケーション内で状態を定義および管理する方法を学びます。
2. シンプルなグラフの構築: 基本的な循環（cyclical）グラフおよび非循環（acyclical）グラフを構築・実行する方法を学びます。
3. ツールの使用の実装: より複雑なタスクのために、グラフ内にツール（関数呼び出し / Function Calling）を統合する方法を学びます。
4. Human-in-the-Loop の組み込み: グラフの実行内に人間のレビューと介入ポイントを追加する方法を学びます。



## 概要

このノートブックでは、大規模言語モデル（LLM）を使用して堅牢でステートフルなエージェント型アプリケーションを構築するためのライブラリである LangGraph について詳しく学びます。LangGraph は LangChain エコシステムを拡張し、ループ、再計画、条件付きステップ実行など、多くのエージェント的動作に不可欠な循環グラフを作成できるようにします。

通常、有向非巡回グラフ（DAG）である従来の LangChain チェーンとは異なり、LangGraph ではノードを再訪し、ステップ間で状態を明示的に管理できる、より複雑なフローを定義できます。これは、計画、アクションの実行、結果の観察、行動の適応を行う AI エージェントのような高度なアプリケーションを構築する上で不可欠です。

このノートブックでは、LangGraph の基本概念（中核コンポーネントであるグラフの状態の理解から、シンプルなグラフの構築、機能拡張のためのツールの統合、LLM アプリケーションへの人間の監視の組み込みまで）を段階的に説明します。

詳細については、公式の [LangGraph documentation](https://langchain-ai.github.io/langgraph/) を参照してください。

## 最もシンプルなグラフ

3 つのノードと 1 つの条件付きエッジを持つシンプルなグラフを構築してみましょう。

![Screenshot 2024-08-20 at 3.11.22 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dba5f465f6e9a2482ad935_simple-graph1.png)

### 状態（State）

[State](https://langchain-ai.github.io/langgraph/concepts/low_level/#state) は、LangGraph における極めて重要な概念です。これは、グラフ内のすべてのノード（ステップ）間で受け渡される単一の統合オブジェクトを表します。アプリケーションのメモリまたは作業コンテキストと考えてください。グラフの実行中、各ノードはこの状態の読み取りと書き込みを行うことができ、ワークフロー全体で情報を蓄積、変更、および意思決定に使用できます。


State により以下が可能になります。
* 情報の永続化: あるステップで生成されたデータ（LLM の応答、ツールの出力など）を以降のステップで利用できるようにします。
* 共有メモリ: グラフ内のすべてのノードが同じ状態オブジェクトに対して動作します（特定のノードがその一部のみを対象とする場合もあります）。これにより一貫性が確保されます。
* 意思決定: 現在の状態を条件付きエッジで使用して、グラフ内の実行フローをルーティングできます。たとえば、エージェントは問題の現在の状態に基づいて次に使用するツールを決定できます。
* 循環フロー: ステップをループまたは再訪するグラフ（LangGraph の強み）において、状態によってゼロからやり直すのではなく、前回のイテレーションを基に各イテレーションを進めることができます。


### 状態スキーマ（State Schema）の定義
State の構造である「スキーマ」は、グラフが追跡する情報の種類を定義します。このスキーマは、グラフ内のすべてのノードの入力タイプとして、また多くのノードの出力タイプとして使用されます。LangGraph は、このスキーマを柔軟に定義できます。このラボでは、Pydantic の BaseModel を使用してグラフの状態を定義します。これにより、構造化、バリデーション、使いやすさのバランスが取れます。LangGraph の State は TypedDict オブジェクトの使用もサポートしています。

In [ ]:
from pydantic import BaseModel


class State(BaseModel):
    graph_state: str

### ノード（Nodes）

LangGraph において、[Nodes](https://langchain-ai.github.io/langgraph/concepts/low_level/#nodes) は計算の基本単位です。グラフ内の個々のステップ、アクション、または処理ロジックを表します。各ノードは、LLM の呼び出し、ツールの実行、データの処理、人間の入力の待機など、特定のタスクの実行を担当します。

##### ノードは Python 関数
LangGraph の優れた側面の 1 つは、ノード定義のシンプルさです。ノードは単なる Python 関数です。この設計アプローチにはいくつかの利点があります。

LangGraph グラフ内でノードとして指定するすべての関数は、最初の位置引数として現在の State オブジェクトを受け取ります。この State オブジェクトは、先ほど定義したスキーマ（Pydantic の BaseModel など）のインスタンスです。このシンプルなグラフでは、ノードは State の `graph_state` フィールドを更新します。

In [ ]:
def node_1(state: State):
    print("---Node 1---")
    return {"graph_state": state.graph_state + " I am"}


def node_2(state: State):
    print("---Node 2---")
    return {"graph_state": state.graph_state + " happy!"}


def node_3(state: State):
    print("---Node 3---")
    return {"graph_state": state.graph_state + " sad!"}

ノード関数を定義したら、次のステップは実際のグラフを構築することです。これは `StateGraph` クラスを使用して行います。

`StateGraph` をインスタンス化する際、定義した State スキーマ（Pydantic の BaseModel や TypedDict など）をコンストラクタに渡します。これにより、内部状態の構造がどのようなものかをグラフに伝え、すべてのノードとエッジが一貫したデータモデルに基づいて動作することを保証します。

グラフオブジェクト（ここでは `builder`）が作成されたら、`add_node()` メソッドを使用して Python 関数をこのグラフ内のノードとして登録します。`add_node()` の各呼び出しは、主に 2 つの引数を取ります。

1. ノードの一意の文字列名（例: "node_1"）。この名前は、エッジと実行フローを定義する際にノードを参照するために使用されます。
2. ノードのロジックを実装する Python 関数自体（例: `node_1`）。

In [ ]:
from IPython.display import Image
from IPython.display import display
from langgraph.graph import END
from langgraph.graph import START
from langgraph.graph import StateGraph

# グラフの構築
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

### エッジ（Edges）

ノード（処理を実行する関数）と定義された状態（フローを通過するデータ）が揃ったので、これらを接続する必要があります。LangGraph におけるエッジはノード間の経路を定義し、操作の順序とアプリケーションの全体的なロジックを指示します。これにより、どのノードの後にどのノードが実行されるかが決まります。

LangGraph は、固定遷移用の通常のエッジと、動的ルーティング用の条件付きエッジを中心に、これらの接続を定義するいくつかの方法を提供します。

#### START および END ポイント
特定のエッジタイプを詳しく見る前に、2 つの特別なキーワードを知っておくことが重要です。

* START: グラフのエントリポイントを定義するために使用される特別な識別子です。通常、START から初期処理ノードへのエッジを追加します。
* END: もう 1 つの特別な識別子です。グラフが END に遷移すると、特定の実行パスが完了したことを示します。グラフは END につながる複数のパスを持つことができます。


#### 通常のエッジ（Normal Edges）
[Normal Edge](https://langchain-ai.github.io/langgraph/concepts/low_level/#edges) は、あるノードから別のノードへの直接的で無条件のリンクを作成します。ノード A からノード B への通常のエッジがある場合、ノード A の実行が終了するたびに、次にノード B が実行されます。

通常のエッジは、`builder.add_edge(source_node_name: str, destination_node_name: str)` メソッドを使用して作成します。

たとえば、特定の処理ステップの後に常に結果を保存したい場合や、分岐が最終アクションにつながる場合に、そのアクションのノードを END に接続します。

#### 条件付きエッジ: 状態に基づく動的ルーティング（Conditional Edges: Dynamic Routing Based on State）
[Conditional Edges](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges) を使用すると、グラフの現在の State に基づいて実行フローをルーティングできます。固定された次のステップの代わりに、関数（「条件」）が評価され、その出力によって次にどのノードを訪問するかが決定されます。

条件付きエッジは次のように実装されます。

* 条件関数: 現在の State オブジェクトを引数として受け取る Python 関数を作成します。
* 判定ロジック: この関数内で状態を検査し、グラフが次にどこへ進むべきかを決定するロジックを実装します。
* 戻り値: 関数は通常、次に実行するノード名である文字列（または文字列のリスト）を返します。

条件付きエッジは、`builder.add_conditional_edges(source_node_name: str, condition_function: Callable, path_map: Optional[dict] = None)` メソッドを使用して追加します。

* `source_node_name`: 条件分岐の起点となるノード。
* `condition_function`: 状態を受け取り、次のノード名を返す Python 関数。
* `path_map`（オプション）: 条件関数がノード名を直接返さない場合（例: 'category_a' や 'category_b' を返す場合）、これらの戻り値を実際のノード名にマッピングする辞書を提供できます（例: {"category_a": "node_for_a", "category_b": "node_for_b"}）。ただし、以下の例で見るように、条件関数が次のノードの文字列表現を直接返す場合は、複雑な path_map は不要です。

以下のコードでは、状態を参照し（このシンプルな例では説明目的でランダムに選択します）、訪問する次のノードの名前を返す `decide_mood` 関数を定義します。この関数は、`node_1` から `node_2` または `node_3` のいずれかへ条件付きでルーティングするために使用されます。

In [ ]:
import random
from typing import Literal


def decide_mood(state) -> Literal["node_2", "node_3"]:

    # 通常は、状態を使用して次に訪問するノードを決定します
    user_input = state.graph_state

    # ここでは、ノード 2 と 3 を 50/50 で分岐させます
    if random.random() < 0.5:

        # 50% の確率で Node 2 を返す
        return "node_2"

    # 50% の確率で Node 3 を返す
    return "node_3"


In [ ]:
# ロジックの設定
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

### コンパイル（Compilation）
グラフを実行する前に、StateGraph の builder インスタンスで `.compile()` メソッドを呼び出す必要があります。このステップにより、グラフの構造が確定し、整合性が検証され、実行可能な LangChain Runnable が返されます。

グラフを可視化することもできます。条件付きエッジは点線で表示されることに注意してください。

In [ ]:
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### グラフの呼び出し（Graph Invocation）

State を定義し、Nodes（処理を行う関数）を追加し、Edges（フローを定義）で接続してコンパイルすると、グラフの設計図が完成します。次のステップは、それを実行することです。

コンパイルされた LangGraph グラフは、[LangChain Runnable](https://python.langchain.com/docs/concepts/runnables/) プロトコルに準拠しています。これは LangChain エコシステム全体で使用される標準インターフェースであり、さまざまなコンポーネントと対話するための統一された方法を提供します。LangChain に精通している場合、`invoke()`、`stream()`、`batch()`、およびそれらの非同期版（`ainvoke()` など）といったメソッドは自然に理解できるでしょう。この標準化により、LangGraph をより大規模な LangChain アプリケーションに容易に統合できます。

##### グラフの実行
グラフを 1 回同期実行するには、`invoke()` メソッドを使用します。`invoke()` メソッドは単一の引数を受け取ります。これは、グラフの**初期状態（initial state）**を定義する辞書です。この辞書のキーは、State スキーマで定義したフィールドに対応している必要があります。たとえば、State オブジェクトに `graph_state` という名前のフィールドがある場合、`invoke` への入力は `{"graph_state": "initial value for graph_state"}` のようになります。

##### 実行プロセス

* 初期化（Initialization）: `invoke()` が呼び出されると、渡された入力辞書によってグラフの初期状態が設定されます。
* 開始地点（Starting Point）: エッジの追加時に定義した特別な START ノードから実行が始まります。
* ノードの実行と状態の更新（Node Execution & State Updates）: 続いて、グラフは最初の実際のノード（今回の例では `node_1`）に遷移します。このノード関数が実行され、現在の状態を受け取ります。ロジックを実行し、更新内容の辞書を返します。これらの更新はグラフの状態に適用されます（デフォルトでは、返されたキーの値を上書きします）。
* エッジの走査（Edge Traversal）: ノードの完了後、グラフはエッジを参照して次のステップを決定します。
* * 通常のエッジがある場合、指定された次のノードに進みます。
* * 条件付きエッジがある場合（`decide_mood` 関数を使用した `node_1` からのエッジなど）、現在の状態を引数として条件関数が呼び出されます。この関数の戻り値によって、次に訪問するノードが決まります（例: 50/50 のロジックに基づいて `node_2` または `node_3`）。
* 継続（Continuation）: ノードの実行、状態の更新、エッジの走査というこのプロセスが継続します。ステップごとに状態が進化し、グラフ全体に情報とコンテキストが伝達されます。
* 終了（Termination）: いずれかのパスが特別な END ノードに到達するまで実行が継続します。END に到達すると、グラフのその特定の実行パスが終了します。

`invoke()` メソッドは、END ノードに到達した後のグラフの**最終状態（final state）**を返します。この最終状態には、実行された一連のノードから蓄積および変更されたすべてのデータが含まれます。

In [ ]:
graph.invoke({"graph_state": ""})

## ツール呼び出し（Function Calling）を備えたシンプルなエージェント

実際の多くのアプリケーションでは、LLM は以下を行う必要があります。
* 最新情報へのアクセス（ウェブ検索、データベースへのクエリなど）。
* 正確な計算の実行（これから確認する算術演算など）。
* 他の API や外部システムとの連携。

Function Calling（または Tool Use: ツールの使用）により、LLM はこのような外部機能をいつ呼び出す必要があるかを示すことができます。LangGraph の構造内では、以下が可能です。
* 現在の状態とタスクに基づいて、ツールを呼び出すかどうか、およびどのツールを呼び出すかを決定するノード（多くの場合 LLM）を配置する。
* そのツールを実行する（別のノードまたは外部の Python 関数）。
* ツールの出力をグラフの状態にフィードバックし、後続のノード（再び LLM など）がこれらの結果を処理してタスクを継続できるようにする。

このセクションでは、ツール呼び出し機能を統合し、ツールをいつ使用するかを決定してその結果をワークフローに組み込むことができるシンプルなエージェントにグラフを変換する方法を示します。これは、より基本的な Function Calling のセットアップにおいて、モデルがツールの呼び出しを選択し、その結果を含む ToolMessage を処理する方法と同様です。
 
![Screenshot 2024-08-21 at 12.45.43 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0b4a2c1e5e02f3e78b_agent2.png)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", vertexai=True)

関数 / ツールを Python 関数として定義します。

In [ ]:
def multiply(a: int, b: int) -> int:
    """a と b を掛け算します。

    Args:
        a: 1 つ目の整数
        b: 2 つ目の整数
    """
    return a * b


def add(a: int, b: int) -> int:
    """a と b を足し算します。

    Args:
        a: 1 つ目の整数
        b: 2 つ目の整数
    """
    return a + b


def divide(a: int, b: int) -> float:
    """a を b で割り算します。

    Args:
        a: 1 つ目の整数
        b: 2 つ目の整数
    """
    return a / b


tools = [add, multiply, divide]

llm_with_tools = llm.bind_tools(tools)

#### アシスタントノードの定義

ここで必要なものは以下のとおりです。
* MessagesState を備えた State: 会話を追跡するために `MessagesState` を使用します。これは `messages` キーの下にメッセージのリスト（人間、AI、ツール）を保持し、新しいメッセージが追加されて履歴が維持されます。定義するアシスタントノードはこれを活用します。

* アシスタントノード: このノードは、関数を呼び出す機能を備えた LLM（`llm_with_tools`）を格納します。現在の `MessagesState` を受け取り、システムプロンプトを含めて応答を生成します。この応答は、直接の回答である場合もあれば、ツールの使用リクエストである場合もあります。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState

# システムメッセージ
sys_msg = SystemMessage(
    content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
)


# ノード
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

#### ツール使用のためのエージェント型ループ（Agentic Loop）
ツールを機能させるために、ループを持つグラフを作成します。

1. アシスタントが実行されます。
2. 条件付きエッジがアシスタントの出力をチェックします。
* ツール呼び出しの場合、フローはツール実行ノード（Tool Executor: 実際のツール関数を実行するように定義）に進みます。
* 直接の回答の場合、フローは END に進みます。
3. ツール実行ノードが実行され、ツールの結果が（ToolMessage として）状態に追加された後、グラフはアシスタントに戻ります。

このループ（assistant -> conditional check -> Tool Executor -> assistant）により、エージェントはツールを呼び出し、その出力を確認した上で、次のアクション（別のツールの呼び出し、または最終的な応答の提示）を決定できます。アシスタントがツール呼び出しではない応答を生成した時点でプロセスは停止します。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(MessagesState)

# ノードの定義: 実際の処理を実行
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# エッジの定義: 制御フローの遷移方法を決定
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # アシスタントからの最新メッセージ（結果）がツール呼び出しの場合 -> tools_condition が tools にルーティング
    # アシスタントからの最新メッセージ（結果）がツール呼び出しでない場合 -> tools_condition が END にルーティング
    tools_condition,
)
builder.add_edge("tools", "assistant")

In [ ]:
react_graph = builder.compile()
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3.")}

# thread = {"configurable": {"thread_id": "1"}}

for event in react_graph.stream(initial_input, stream_mode="values"):
    event["messages"][-1].pretty_print()

## ブレークポイントと Human-in-the-loop

自動化されたエージェントは強力ですが、人間の監視や介入のためのポイントが必要になることがあります。LangGraph はブレークポイントとチェックポインティングを通じてこれを容易にします。

ブレークポイントを設定することで、特定のノードが実行される直前にグラフの実行を一時停止できます。この一時停止により、人間は現在の状態を確認し、必要に応じて変更したり、エージェントが次に意図するアクションを検証してから続行を許可したりできます。

これを有効にするために、以下を使用します。

* チェックポインター（`MemorySaver` など）: 中断時を含め、さまざまなポイントでグラフの状態を保存します。これにより、後から実行を再開できるようになります。
* グラフコンパイル時の `interrupt_before` 引数: 実行前に中断をトリガーするノードを LangGraph に指示します。

以下のコードでは、チェックポインターを使用してグラフをコンパイルし、中断ポイントを設定することで、フローを検査し、Human-in-the-loop のシナリオをシミュレートする方法を示します。

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

react_graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

thread = {"configurable": {"thread_id": "1"}}

for event in react_graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

前のステップでは、チェックポインティングのために `MemorySaver` を使用してグラフをコンパイルし、`interrupt_before=["tools"]` を指定しました。これにより、"tools" ノード（ツール実行ノード）が通常実行される直前に実行を一時停止するようグラフに指示されます。また、`thread_id: "1"` を使用してストリームを開始しました。そのストリーム中、グラフは入力を処理し、"tools" ノードに到達した時点で一時停止し、`thread_id: "1"` に関連付けられた現在の状態をメモリチェックポインターに保存しました。

ここでは、より明示的な Human-in-the-loop のシナリオを示します。具体的には次の手順を行います。

* 新しい `thread_id: "2"` を使用して、独立した新しい実行ストリームを開始します。これにより、複数のグラフ実行とその状態を個別に管理できます。
* この新しいストリームを、同じ `interrupt_before=["tools"]` ブレークポイントに到達するまで実行します。
* この一時停止時に、入力を促すプロンプトを表示して人間による意思決定をシミュレートします。
* この入力に基づいて、（承認された場合は）`thread_id: "2"` のグラフの実行を中断した場所から再開するか、このスレッドの以降の処理を停止します。

これは、エージェントのワークフローに介入し、その意図を検査して、アクションを誘導する方法を示しています。

In [ ]:
# スレッドの設定
thread = {"configurable": {"thread_id": "2"}}

# 最初の中断までグラフを実行
for event in react_graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

# ユーザーからのフィードバックを取得
user_approval = input("Do you want to call the tool? (yes/no): ")

# 承認を確認
if user_approval.lower() == "yes":

    # 承認された場合、グラフの実行を継続
    for event in react_graph.stream(None, thread, stream_mode="values"):
        event["messages"][-1].pretty_print()

else:
    print("Operation cancelled by user.")

---
## 複雑なエッセイ執筆エージェントの構築

このセクションでは、Gemini を使用してエッセイの生成、推敲、批評を行う LangGraph 搭載の AI エージェントを構築する方法を説明します。この LangGraph コードは、DeepLearning.AI の優れたコース「AI Agents in LangGraph」から応用したものです。

「Planner（計画）」、「Research Plan（調査計画）」、「Generate（生成）」、「Reflect（省察 / 批評）」、「Research Critique（批評の調査）」などのノードを持つ構造化された状態フローを定義することで、システムは指定されたトピックに関するエッセイを反復的に作成し、フィードバックを取り入れ、リサーチに基づいた洞察を提供します。

エージェントは次のステップを実行します。
1.  **Plan（計画）:** 指定されたトピックに基づいてエッセイの初期アウトラインを作成します。
3.  **Generate（生成）:** 計画と調査内容を使用してエッセイのドラフトを執筆します。
4.  **Reflect（省察 / 批評）:** 生成されたドラフトを批評し、改善点を特定します。
6.  **Loop（ループ）:** 生成、省察、批評調査のステップを指定回数ループさせて、エッセイを段階的に改善します。

このワークフローにより、改訂管理を備えた自動エッセイ生成が可能になり、構造化された執筆タスクや教育的なユースケースに最適です。さらに、このノートブックでは外部検索ツールを使用して、リアルタイム情報を収集しエッセイの内容に統合します。

In [ ]:
import os
from typing import List
from typing import Optional

from IPython.display import HTML
from IPython.display import Markdown
from IPython.display import display

### Google Search を使用した Google Gen AI SDK の設定
このエージェントフローでは、LangChain に加えて Google Search の機能を簡単に活用できるように Gen AI SDK も使用します。

In [ ]:
from google import genai
from google.genai import types

# クライアントの設定
client = genai.Client(vertexai=True, location="global")

# グラウンディングツールの定義
grounding_tool = types.Tool(google_search=types.GoogleSearch())

# 生成設定の構成
config = types.GenerateContentConfig(
    tools=[grounding_tool],
)

### エージェントのメモリ、エージェントの状態、および検索クエリ用スキーマの初期化

エージェントのノードとグラフ構造を定義する前に、基礎となるコンポーネントをセットアップする必要があります。
* **エージェントの状態（`AgentState`）:** この Pydantic の `BaseModel` は、グラフ内を流れるすべての情報を保持します。これには、タスク、エッセイ計画、ドラフト、批評、改訂番号、および許可される最大改訂回数が含まれます。グラフ内の各ノードは、この状態の読み取りと書き込みを行います。

In [ ]:
class AgentState(BaseModel):
    task: str
    plan: Optional[str] = ""
    draft: Optional[str] = ""
    critique: Optional[str] = ""
    revision_number: int
    max_revisions: int

### 各ステージのプロンプトテンプレートの定義

エッセイ執筆プロセスの各ステージで LLM を適切に誘導するためには、明確で効果的なプロンプトが不可欠です。以下の特定のプロンプトテンプレートを定義します。
* **PLAN_PROMPT:** 専門の執筆者として行動し、エッセイのハイレベルなアウトラインを作成するよう LLM に指示します。
* **WRITER_PROMPT:** エッセイアシスタントとして行動し、計画と調査に基づいて 3 ページのエッセイを執筆し、批評に基づいて推敲するよう LLM に指示します。また、Markdown 形式の指定も行います。
* **REFLECTION_PROMPT:** エッセイを採点する教師として行動し、批評と詳細な推奨事項を提供するよう LLM に指示します。

In [ ]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay.
Write such an outline for the user provided topic. Give an outline of the essay along with any
relevant notes or instructions for the sections."""

WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 3-pages essays.
First, gather information using the search tool to write a great essay.
Then, generate the best essay possible for the user's request.
If the user provides critique , respond with a revised version of your previous attempts.
Use Markdown formatting to specify a title and section headers for each paragraph.
"""

REFLECTION_PROMPT = """You are a teacher grading an essay submission.
Generate critique and recommendations for the user's submission.
Provide detailed recommendations, including requests for length, depth, style, etc.
Also, provide suggetion about additional information the writer should gather."""

### 各ステージのノード関数の定義

LangGraph 内の各ノードは、エッセイ執筆ワークフローにおける特定のステップを表します。これらの Python 関数は現在の `AgentState` を入力として受け取り、それぞれの処理に対して更新された状態値を含む辞書を返します。

* `plan_node`: `PLAN_PROMPT` を使用して LLM を呼び出し、エッセイのアウトラインを生成します。
* `generation_node`: `WRITER_PROMPT` を使用して LLM を呼び出します（情報収集のために Google Search ツールを呼び出します）。これによりエッセイのドラフトを生成し、改訂番号をインクリメントします。
* `reflection_node`: `REFLECTION_PROMPT` を使用して LLM で現在のドラフトを批評します。
* `should_continue`: これは条件付きエッジ関数です。現在の `revision_number` が `max_revisions` を超えたかどうかを確認します。超えている場合は `END` を返してグラフを終了し、そうでない場合は "reflect" を返して推敲サイクルを継続します。

In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state.task),
    ]
    response = llm.invoke(messages)
    
    if isinstance(response.content, list):
        plan_text = response.content[0]["text"]
    else:
        plan_text = response.content

    
    return {"plan": plan_text}


# 内容と計画に基づいてドラフトを生成
def generation_node(state: AgentState):
    # 生成設定の構成
    config = types.GenerateContentConfig(
        tools=[grounding_tool], system_instruction=WRITER_PROMPT
    )
    contents = f"Task: {state.task}, critique: {state.critique}"

    # リクエストの実行
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=contents,
        config=config,
    )

    # 検索クエリの表示
    if response.candidates[0].grounding_metadata is not None:
        display(
            Markdown(
                "**Generator is gathering information with these search queries...:**"
            )
        )
        display(
            HTML(
                response.candidates[
                    0
                ].grounding_metadata.search_entry_point.rendered_content
            )
        )

    return {
        "draft": response.text,
        "revision_number": state.revision_number + 1,
    }


# ドラフトに対するフィードバックや批評を提供
def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state.draft),
    ]
    response = llm.invoke(messages)

    if isinstance(response.content, list):
        critique_text = response.content[0]["text"]
    else:
        critique_text = response.content
    
    return {"critique": critique_text}


# 改訂回数に基づいて批評と調査のサイクルを
# 継続するかどうかを決定
def should_continue(state):
    if state.revision_number >= state.max_revisions:
        return END
    return "reflect" 

### グラフの定義とコンパイル

状態、プロンプト、ノード関数を定義したので、エッセイ執筆エージェントのグラフを構築します。
1.  **`StateGraph` の初期化:** `AgentState` を使用して `StateGraph` のインスタンスを作成します。
2.  **ノードの追加:** 前のステップで定義した各関数（`plan_node`、`generation_node` など）を一意の文字列識別子とともにグラフにノードとして追加します。
3.  **エントリポイントの設定:** `builder.set_entry_point("planner")` により、グラフの開始点として `planner` ノードを指定します。
4.  **エッジの追加:**
    * 順次エッジ: メインフロー（`planner` -> `generate`）と、推敲ループ（`reflect` -> `generate`）を定義します。
    * 条件付きエッジ: `generate` ノードの後に `builder.add_conditional_edges` を使用します。これにより `should_continue` 関数が呼び出されます。その戻り値に基づいて、グラフは `reflect` ノード（推敲の継続）または `END`（完了）のいずれかに遷移します。
5.  **メモリの初期化:** グラフの状態を保存するチェックポインティングを有効にするため、`MemorySaver` を初期化します。
6.  **グラフのコンパイル:** `builder.compile(checkpointer=memory)` によりグラフ構造を確定し、メモリセーバーを組み込んで、グラフを実行可能な状態にします。

In [ ]:
# 状態グラフの初期化
builder = StateGraph(AgentState)

# ワークフローの各ステップのノードを追加
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)

# ワークフローのエントリポイントを設定
builder.set_entry_point("planner")

# タスクの継続または終了のための条件付きエッジを追加
builder.add_conditional_edges(
    "generate", should_continue, {END: END, "reflect": "reflect"}
)

# タスク順序のエッジを定義
builder.add_edge("planner", "generate")

builder.add_edge("reflect", "generate")

# エージェントメモリの初期化
memory = MemorySaver()

# メモリ状態管理を使用してグラフをコンパイル
graph = builder.compile(checkpointer=memory)

In [ ]:
Image(graph.get_graph().draw_mermaid_png())

### エージェントの実行 - 執筆の開始！

それでは、エッセイ執筆エージェントを実行してみましょう。最初のエッセイトピックと設定を指定し、エージェントの実行をストリーミングして各ステップとその出力を観察します。

In [ ]:
# エッセイのトピックを定義
ESSAY_TOPIC = (
    "What were the impacts of Hurricane Helene and Hurricane Milton in 2024?"
)

# 一意のスレッド ID を使用してスレッド設定を定義
thread = {"configurable": {"thread_id": "1"}}

# 初期タスクと初期状態を指定してグラフの実行をストリーミング
for s in graph.stream(
    {
        "task": ESSAY_TOPIC,  # 初期タスク
        "max_revisions": 3,  # 許可される最大改訂回数
        "revision_number": 0,  # 現在の改訂番号
    },
    thread,
):
    step = next(iter(s))
    display(Markdown(f"# {step}"))
    for key, content in s[step].items():
        if key == "revision_number":
            display(Markdown(f"**Revision Number**: {content}"))
        elif isinstance(content, list):
            for c in content:
                display(Markdown(c))
        else:
            display(Markdown(content))
    print("\n---\n")

### LangGraph についてさらに詳しく学ぶ
さらに深く学びたい場合は、公式の LangGraph ドキュメントが最適なリソースです。

* [Core Concepts](https://langchain-ai.github.io/langgraph/): State、Nodes、Edges などの基本要素の復習。
* [Tutorials](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/): さまざまなサンプルで LangGraph の実際の動作を確認。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.